# Lunar Forzen Orbits

**References**

[1] T. A. Ely, ‘Stable Constellations of Frozen Elliptical Inclined Lunar Orbits’, J of Astronaut Sci, vol. 53, no. 3, pp. 301–316, Sep. 2005, doi: 10.1007/BF03546355.


In [1]:
import pylupnt as pnt
import numpy as np

np.set_printoptions(precision=3, suppress=True)

In [2]:
# Starting epoch of 1-July-2009 1:00:00
# International Atomic Time (TAI) in seconds
t0 = pnt.gregorian2time(2009, 7, 1, 1, 0, 0)

# Classical Orbital Elements (COE)
# Earth's apparent orbital plane (OP) frame
a = 6541.4  # [km] Semi-major axis
e = 0.60  # [-] Eccentricity
i = 56.2  # [deg] Inclination
O = 0.00 * pnt.RAD  # [deg] Right ascension of the ascending node
w = 90.0 * pnt.RAD  # [deg] Argument of perigee
M = 0.00 * pnt.RAD  # [deg] Mean anomaly

coe0 = np.array([a, e, i, O, w, M])
rv0 = pnt.classical2cart(coe0, pnt.GM_MOON)

print("coe0:", coe0, " [km, -, rad, rad, rad, rad]")
print("rv0: ", rv0, " [km, km, km, km/s, km/s, km/s]")

coe0: [6541.4      0.6     56.2      0.       1.571    0.   ]  [km, -, rad, rad, rad, rad]
rv0:  [   0.    2459.118 -893.937   -1.731    0.      -0.   ]  [km, km, km, km/s, km/s, km/s]


## Case 1
Two-year propagation with Earth as the only perturbation. The orbit of the Earth is exactly circular, and all the results are reconciled in the Earth orbit plane.

In [3]:
moon_period = 2 * np.pi * np.sqrt(pnt.D_EARTH_MOON ** 3 / pnt.GM_EARTH)
print(f'Moon period: {moon_period / pnt.SECS_DAY:.3f} days')

Moon period: 27.452 days


In [5]:
class Case1Dynamics(pnt.NumericalOrbitDynamics):
    def __init__(self):
        super().__init__(odefunc=self.compute_rates, integ=pnt.IntegratorType.RK4)
    
    def compute_rates(self, t, rv):
        r = rv[:3]
        v = rv[3:]
        a = -self.GM * r / np.linalg.norm(r) ** 3
        return np.zeros((6,1))
        return np.concatenate((v, a))

In [6]:
dyn_case1 = Case1Dynamics()

TypeError: __init__(): incompatible constructor arguments. The following argument types are supported:
    1. pylupnt._pylupnt.NumericalOrbitDynamics(odefunc: std::__1::function<Eigen::Matrix<autodiff::detail::Real<1ul, double>, -1, 1, 0, -1, 1> (autodiff::detail::Real<1ul, double>, Eigen::Matrix<autodiff::detail::Real<1ul, double>, -1, 1, 0, -1, 1> const&)>, integ_type: pylupnt._pylupnt.IntegratorType = <IntegratorType.RK4: 0>)

Invoked with: kwargs: odefunc=<bound method Case1Dynamics.compute_rates of <__main__.Case1Dynamics object at 0x107d4f0b0>>, integ=<IntegratorType.RK4: 0>

Did you forget to `#include <pybind11/stl.h>`? Or <pybind11/complex.h>,
<pybind11/functional.h>, <pybind11/chrono.h>, etc. Some automatic
conversions are optional and require extra headers to be included
when compiling your pybind11 module.

In [22]:
dt_total = 2 * pnt.DAYS_YEAR * pnt.SECS_DAY # [s] Total duration
dt_step = 1 * pnt.SECS_DAY  # [s] Time step
tspan = np.arange(0, dt_total, dt_step) # [s] Time span
tfs = t0 + tspan # [s] Time in TAI

rv0_earth2moon = pnt.get_body_pos_vel(t0, pnt.EARTH, pnt.MOON)
coe0_moon = pnt.cart2classical(rv0_earth2moon, pnt.GM_EARTH)

kep_dyn = pnt.KeplerianDynamics(pnt.GM_EARTH)
coe_moon = kep_dyn.propagate(coe0_moon, t0, tfs)